In [1]:
from pathlib import Path
import random
import shutil

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}


def split_dataset(
    source_root,
    output_root,
    train_ratio=0.7,
    val_ratio=0.15,
    seed=42,
):
    random.seed(seed)

    source_root = Path(source_root)
    output_root = Path(output_root)

    for label in ["real", "fake"]:

        src_dir = source_root / label

        videos = [
            p for p in src_dir.iterdir()
            if p.suffix.lower() in VIDEO_EXTS
        ]

        random.shuffle(videos)

        n = len(videos)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_files = videos[:n_train]
        val_files = videos[n_train:n_train + n_val]
        test_files = videos[n_train + n_val:]

        splits = {
            "train": train_files,
            "val": val_files,
            "test": test_files,
        }

        for split_name, files in splits.items():

            dst_dir = output_root / split_name / label
            dst_dir.mkdir(parents=True, exist_ok=True)

            for file in files:
                shutil.copy2(file, dst_dir / file.name)

            print(
                f"{split_name:5s} | {label:4s} | "
                f"{len(files)} videos"
            )

    print("\nDataset split completed.")

In [2]:
split_dataset(
    source_root="dataset\\FF++",
    output_root="dataset_split",
    train_ratio=0.70,
    val_ratio=0.15,
)

train | real | 140 videos
val   | real | 30 videos
test  | real | 30 videos
train | fake | 140 videos
val   | fake | 30 videos
test  | fake | 30 videos

Dataset split completed.


In [3]:
from pathlib import Path
import cv2


def extract_frames(video_path, output_dir, max_frames=30):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    existing = sorted(output_dir.glob("*.jpg"))
    if len(existing) >= max_frames:
        return [str(p) for p in existing[:max_frames]]

    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        cap.release()
        return []

    indices = [
        int(i * total / max_frames)
        for i in range(max_frames)
    ]

    paths = []

    frame_idx = 0
    save_idx = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_idx in indices:
            path = output_dir / f"frame_{save_idx:04d}.jpg"

            cv2.imwrite(str(path), frame)

            paths.append(str(path))
            save_idx += 1

        frame_idx += 1

    cap.release()

    return paths

ModuleNotFoundError: No module named 'cv2'

In [4]:
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}


def build_video_index(
    split_dir,
    frame_cache_dir,
    max_frames=30,
):
    split_dir = Path(split_dir)

    samples = []

    for label_name, label in [
        ("real", 0),
        ("fake", 1),
    ]:

        video_dir = split_dir / label_name

        videos = [
            p for p in video_dir.iterdir()
            if p.suffix.lower() in VIDEO_EXTS
        ]

        for video_path in videos:

            cache_dir = (
                Path(frame_cache_dir)
                / label_name
                / video_path.stem
            )

            frame_paths = extract_frames(
                video_path,
                cache_dir,
                max_frames=max_frames,
            )

            if len(frame_paths) == 0:
                continue

            samples.append(
                (frame_paths, label)
            )

    return samples

In [ ]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset


class FaceForensicsVideoDataset(Dataset):

    def __init__(
        self,
        samples,
        transform=None,
        seq_len=8,
    ):
        self.samples = samples
        self.transform = transform
        self.seq_len = seq_len

        print(f"Videos: {len(samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        frame_paths, label = self.samples[idx]

        if len(frame_paths) >= self.seq_len:

            indices = np.linspace(
                0,
                len(frame_paths) - 1,
                self.seq_len,
                dtype=int,
            )

            frame_paths = [
                frame_paths[i]
                for i in indices
            ]

        else:

            frame_paths += [
                frame_paths[-1]
            ] * (self.seq_len - len(frame_paths))

        frames = []

        for path in frame_paths:

            img = Image.open(path).convert("RGB")

            if self.transform:
                img = self.transform(img)

            frames.append(img)

        video = torch.stack(frames)

        return video, torch.tensor(label)

In [ ]:
from torch.utils.data import DataLoader


def build_dataloaders(
    dataset_root,
    frame_cache_root="./frame_cache",
    transform=None,
    batch_size=4,
    seq_len=8,
    num_workers=4,
):

    train_samples = build_video_index(
        f"{dataset_root}/train",
        f"{frame_cache_root}/train",
    )

    val_samples = build_video_index(
        f"{dataset_root}/val",
        f"{frame_cache_root}/val",
    )

    test_samples = build_video_index(
        f"{dataset_root}/test",
        f"{frame_cache_root}/test",
    )

    train_dataset = FaceForensicsVideoDataset(
        train_samples,
        transform=transform,
        seq_len=seq_len,
    )

    val_dataset = FaceForensicsVideoDataset(
        val_samples,
        transform=transform,
        seq_len=seq_len,
    )

    test_dataset = FaceForensicsVideoDataset(
        test_samples,
        transform=transform,
        seq_len=seq_len,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader, test_loader